<a href="https://colab.research.google.com/github/nidhi2407032-dotcom/CPOT_TENS-Project/blob/main/AND%2BBiLSTM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

np.random.seed(42)
N = 300

def gen(ranges, n=N):
    return {k: np.random.uniform(lo, hi, n) for k, (lo, hi) in ranges.items()}

emg_ranges = {
    "Normal":  {"emg_mean":(1.99,24.67), "emg_std":(4.48,93.26), "emg_rms":(4.97,96.47),
                "emg_max":(15.0,408.99), "emg_range":(15.57,409.64)},
    "Acute":   {"emg_mean":(2.11,27.50), "emg_std":(4.54,104.81), "emg_rms":(5.01,108.36),
                "emg_max":(15.0,459.49), "emg_range":(15.75,460.65)},
    "Chronic": {"emg_mean":(2.54,27.56), "emg_std":(4.74,105.04), "emg_rms":(5.37,108.60),
                "emg_max":(15.0,460.49), "emg_range":(15.67,461.65)},
}
imu_ranges = {
    "Normal":  {"imu_mean_mag":(3.13,80.51), "imu_std_mag":(0.093,0.35),
                "imu_max_mag":(3.35,80.75), "imu_range":(0.363,1.431), "imu_jerk":(0.009,0.080)},
    "Acute":   {"imu_mean_mag":(2.35,79.75), "imu_std_mag":(0.12,0.85),   # interpolated: no Moderate row was reported
                "imu_max_mag":(3.38,79.99), "imu_range":(0.42,2.5), "imu_jerk":(0.0095,0.15)},
    "Chronic": {"imu_mean_mag":(1.58,79.01), "imu_std_mag":(0.144,1.42),
                "imu_max_mag":(3.41,79.25), "imu_range":(0.489,4.335), "imu_jerk":(0.010,0.285)},
}
gsr_ranges = {
    "Normal":  {"gsr_mean":(0.087,0.133), "gsr_std":(0.0002,0.0012), "gsr_slope":(-0.003,0.001),
                "gsr_max":(0.088,0.134), "gsr_range":(0.001,0.003)},
    "Acute":   {"gsr_mean":(0.087,0.133), "gsr_std":(0.0002,0.0012), "gsr_slope":(-0.004,0.002),
                "gsr_max":(0.089,0.134), "gsr_range":(0.001,0.004)},
    "Chronic": {"gsr_mean":(0.092,0.132), "gsr_std":(0.0003,0.0013), "gsr_slope":(-0.001,0.003),
                "gsr_max":(0.093,0.133), "gsr_range":(0.002,0.004)},
}

def build_dataset(ranges_dict):
    frames = []
    for label, ranges in ranges_dict.items():
        d = pd.DataFrame(gen(ranges))
        d["label"] = label
        frames.append(d)
    return pd.concat(frames, ignore_index=True)

emg_df = build_dataset(emg_ranges)
imu_df = build_dataset(imu_ranges)
gsr_df = build_dataset(gsr_ranges)

emg_feats = ["emg_mean","emg_std","emg_rms","emg_max","emg_range"]
imu_feats = ["imu_mean_mag","imu_std_mag","imu_max_mag","imu_range","imu_jerk"]
gsr_feats = ["gsr_mean","gsr_std","gsr_slope","gsr_max","gsr_range"]

emg_rf = RandomForestClassifier(n_estimators=300, random_state=42).fit(emg_df[emg_feats], emg_df["label"])
imu_rf = RandomForestClassifier(n_estimators=300, random_state=42).fit(imu_df[imu_feats], imu_df["label"])
gsr_rf = RandomForestClassifier(n_estimators=300, random_state=42).fit(gsr_df[gsr_feats], gsr_df["label"])


W = {"emg": 0.679, "imu": 0.909, "gsr": 0.766}
SEV = {"Normal": 0.0, "Acute": 0.5, "Chronic": 1.0}


MAX_MA = 25.0

def classify_and_fuse(vals):
    e = pd.DataFrame([{
        "emg_mean":vals["emg_mean"], "emg_std":vals["emg_std"], "emg_rms":vals["emg_rms"],
        "emg_max":vals["emg_max"], "emg_range":vals["emg_range"]}])
    i = pd.DataFrame([{
        "imu_mean_mag":vals["imu_mean_mag"], "imu_std_mag":vals["imu_std_mag"],
        "imu_max_mag":vals["imu_max_mag"], "imu_range":vals["imu_range"], "imu_jerk":vals["imu_jerk"]}])
    g = pd.DataFrame([{
        "gsr_mean":vals["gsr_mean"], "gsr_std":vals["gsr_std"], "gsr_slope":vals["gsr_slope"],
        "gsr_max":vals["gsr_max"], "gsr_range":vals["gsr_range"]}])

    e_pred, i_pred, g_pred = emg_rf.predict(e)[0], imu_rf.predict(i)[0], gsr_rf.predict(g)[0]
    e_sev, i_sev, g_sev = SEV[e_pred], SEV[i_pred], SEV[g_pred]

    fused = (W["emg"]*e_sev + W["imu"]*i_sev + W["gsr"]*g_sev) / sum(W.values())

    # AND-gate: only escalate label if >=2 modalities agree at Acute+ or Chronic
    votes = [e_pred, i_pred, g_pred]
    if votes.count("Chronic") >= 2:
        final_label = "Chronic"
    elif sum(v in ("Acute","Chronic") for v in votes) >= 2:
        final_label = "Acute"
    else:
        final_label = "Normal"

    cpot_score = round(fused, 3)
    tens_ma = 0.0
    mode = "No stimulation"
    if final_label == "Acute":
        tens_ma = round(8 + (cpot_score-0.34)/(0.66-0.34) * (15-8), 1)
        mode = "Conventional (80-120 Hz, 100 µs)"
    elif final_label == "Chronic":
        tens_ma = round(15 + (cpot_score-0.67)/(1.0-0.67) * (25-15), 1)
        mode = "Low-rate / acupuncture-like (2-10 Hz, 200 µs)"
    tens_ma = min(tens_ma, MAX_MA)  # hard safety ceiling

    return {
        "emg_pred": e_pred, "imu_pred": i_pred, "gsr_pred": g_pred,
        "cpot_score": cpot_score, "pain_class": final_label,
        "tens_intensity_mA": tens_ma, "tens_mode": mode
    }

print("="*70)
print("REFERENCE FEATURE RANGES (Normal / Acute / Chronic)")
print("="*70)
ref = {
 "EMG mean (µV)":      ("1.99-24.67","2.11-27.50","2.54-27.56"),
 "EMG std (µV)":       ("4.48-93.26","4.54-104.81","4.74-105.04"),
 "EMG RMS (µV)":       ("4.97-96.47","5.01-108.36","5.37-108.60"),
 "EMG max (µV)":       ("15.0-408.99","15.0-459.49","15.0-460.49"),
 "EMG range (µV)":     ("15.57-409.64","15.75-460.65","15.67-461.65"),
 "IMU mean mag (°/s)": ("3.13-80.51","~interp.","1.58-79.01"),
 "IMU std mag (°/s)":  ("0.093-0.35","~interp.","0.144-1.42"),
 "IMU max mag (°/s)":  ("3.35-80.75","~interp.","3.41-79.25"),
 "IMU range (°/s)":    ("0.363-1.431","~interp.","0.489-4.335"),
 "IMU jerk (°/s³)":    ("0.009-0.080","~interp.","0.010-0.285"),
 "GSR mean (µS)":      ("0.087-0.133","0.087-0.133","0.092-0.132"),
 "GSR std (µS)":       ("~0.000-0.001","~0.000-0.001","~0.001"),
 "GSR slope (µS/s)":   ("-0.003 to 0.001","-0.004 to 0.002","-0.001 to 0.003"),
 "GSR max (µS)":       ("0.088-0.134","0.089-0.134","0.093-0.133"),
 "GSR range (µS)":     ("0.001-0.003","0.001-0.004","0.002-0.004"),
}
print(f"{'Feature':<20}{'Normal':<18}{'Acute':<18}{'Chronic'}")
for k,(a,b,c) in ref.items():
    print(f"{k:<20}{a:<18}{b:<18}{c}")


print("\n" + "="*70)
print("ENTER PATIENT SENSOR VALUES")
print("="*70)
def ask(name, unit):
    return float(input(f"{name} ({unit}): "))

vals = {
 "emg_mean": ask("EMG mean","µV"), "emg_std": ask("EMG std","µV"),
 "emg_rms": ask("EMG RMS","µV"), "emg_max": ask("EMG max","µV"),
 "emg_range": ask("EMG range","µV"),
 "imu_mean_mag": ask("IMU mean magnitude","°/s"), "imu_std_mag": ask("IMU std magnitude","°/s"),
 "imu_max_mag": ask("IMU max magnitude","°/s"), "imu_range": ask("IMU range","°/s"),
 "imu_jerk": ask("IMU jerk","°/s³"),
 "gsr_mean": ask("GSR mean","µS"), "gsr_std": ask("GSR std","µS"),
 "gsr_slope": ask("GSR slope","µS/s"), "gsr_max": ask("GSR max","µS"),
 "gsr_range": ask("GSR range","µS"),
}

result = classify_and_fuse(vals)

print("\n" + "="*70)
print("RESULT")
print("="*70)
print(f"EMG modality prediction : {result['emg_pred']}")
print(f"IMU modality prediction : {result['imu_pred']}")
print(f"GSR modality prediction : {result['gsr_pred']}")
print(f"\nFused CPOT score (0-1)  : {result['cpot_score']}")
print(f"Pain classification     : {result['pain_class']}")
print(f"Recommended TENS output : {result['tens_intensity_mA']} mA  |  Mode: {result['tens_mode']}")
print(f"(Hard safety ceiling: {MAX_MA} mA — requires manual confirm before hardware applies it)")

REFERENCE FEATURE RANGES (Normal / Acute / Chronic)
Feature             Normal            Acute             Chronic
EMG mean (µV)       1.99-24.67        2.11-27.50        2.54-27.56
EMG std (µV)        4.48-93.26        4.54-104.81       4.74-105.04
EMG RMS (µV)        4.97-96.47        5.01-108.36       5.37-108.60
EMG max (µV)        15.0-408.99       15.0-459.49       15.0-460.49
EMG range (µV)      15.57-409.64      15.75-460.65      15.67-461.65
IMU mean mag (°/s)  3.13-80.51        ~interp.          1.58-79.01
IMU std mag (°/s)   0.093-0.35        ~interp.          0.144-1.42
IMU max mag (°/s)   3.35-80.75        ~interp.          3.41-79.25
IMU range (°/s)     0.363-1.431       ~interp.          0.489-4.335
IMU jerk (°/s³)     0.009-0.080       ~interp.          0.010-0.285
GSR mean (µS)       0.087-0.133       0.087-0.133       0.092-0.132
GSR std (µS)        ~0.000-0.001      ~0.000-0.001      ~0.001
GSR slope (µS/s)    -0.003 to 0.001   -0.004 to 0.002   -0.001 to 0.003
GSR 